In [29]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import warnings

In [30]:
data = pd.read_csv('data/r5_eda.csv')

In [31]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6497 entries, 0 to 6496
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed_acidity         6497 non-null   float64
 1   volatile_acidity      6497 non-null   float64
 2   citric_acid           6497 non-null   float64
 3   residual_sugar        6497 non-null   float64
 4   chlorides             6497 non-null   float64
 5   free_sulfur_dioxide   6497 non-null   int64  
 6   total_sulfur_dioxide  6497 non-null   int64  
 7   density               6497 non-null   float64
 8   pH                    6497 non-null   float64
 9   sulphates             6497 non-null   float64
 10  alcohol               6497 non-null   float64
 11  quality               6497 non-null   int64  
dtypes: float64(9), int64(3)
memory usage: 609.2 KB


In [32]:
y = data['quality']
X = data.drop('quality', axis=1)

In [33]:
from sklearn.model_selection import train_test_split

In [34]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [35]:
X_train.shape, y_train.shape, X_test.shape, y_test.shape

((5197, 11), (5197,), (1300, 11), (1300,))

In [36]:
from sklearn.linear_model import LinearRegression, Lasso, Ridge

## Простая линейная регрессия

In [37]:
lr = LinearRegression().fit(X_train, y_train)

In [38]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error, r2_score
from math import sqrt

In [39]:
y_pred = lr.predict(X_test)

In [40]:
print(f'MAE: {mean_absolute_error(y_test, y_pred)}')
print(f'MSE: {mean_squared_error(y_test, y_pred)}')
print(f'RMSE: {sqrt(mean_squared_error(y_test, y_pred))}')
print(f'MAPE: {sqrt(mean_absolute_percentage_error(y_test, y_pred))}')
print(f'R^2: {r2_score(y_test, y_pred)}')

MAE: 0.5761787489910857
MSE: 0.5509038732744104
RMSE: 0.7422289897830793
MAPE: 0.32190330139337814
R^2: 0.2681170680632926


In [41]:
len(lr.coef_)
lr.coef_

array([ 7.11238538e-02, -1.22805081e+00, -9.60427170e-02,  4.53283006e-02,
       -6.00073619e-01,  6.67776645e-03, -2.48229218e-03, -5.64276856e+01,
        4.89000042e-01,  7.51555348e-01,  2.73259137e-01])

## Линейная регрессия с L1-регуляризацией

In [42]:
ridge = Ridge(alpha=0.5).fit(X_train, y_train)
y_pred = ridge.predict(X_test)
print(f'MAE: {mean_absolute_error(y_test, y_pred)}')
print(f'MSE: {mean_squared_error(y_test, y_pred)}')
print(f'RMSE: {sqrt(mean_squared_error(y_test, y_pred))}')
print(f'MAPE: {sqrt(mean_absolute_percentage_error(y_test, y_pred))}')
print(f'R^2: {ridge.score(X_test, y_test)}')
ridge.coef_

MAE: 0.5779000001790205
MSE: 0.5519556606037792
RMSE: 0.7429371848304399
MAPE: 0.3222754442059757
R^2: 0.2667197549715967


array([ 0.01346997, -1.35675326, -0.10068265,  0.02399808, -0.80773652,
        0.00670503, -0.00222475, -0.34317743,  0.20816111,  0.62171547,
        0.33888815])

### Подбор гиперпараметров

In [43]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, cross_val_score
import numpy as np
import optuna

p:\Python\ML1\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


GridSearchCV

In [44]:
param_grid = {'alpha': [0.001, 0.01, 0.1, 0.5, 1, 10]}

In [45]:
grid = GridSearchCV(Lasso(), param_grid, cv=5, scoring='r2')
grid.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=Lasso(),
             param_grid={'alpha': [0.001, 0.01, 0.1, 0.5, 1, 10]},
             scoring='r2')

In [46]:
print("Best alpha:", grid.best_params_['alpha'])
print("Best R^2:", grid.best_score_)

Best alpha: 0.001
Best R^2: 0.2895408021415575


RandomizedSearchCV

In [47]:
param_dist = {'alpha': np.logspace(-4, 1, 100)}

In [48]:
random_search = RandomizedSearchCV(Lasso(), param_dist, n_iter=20, cv=5, scoring='r2', random_state=42)
random_search.fit(X_train, y_train)

RandomizedSearchCV(cv=5, estimator=Lasso(), n_iter=20,
                   param_distributions={'alpha': array([1.00000000e-04, 1.12332403e-04, 1.26185688e-04, 1.41747416e-04,
       1.59228279e-04, 1.78864953e-04, 2.00923300e-04, 2.25701972e-04,
       2.53536449e-04, 2.84803587e-04, 3.19926714e-04, 3.59381366e-04,
       4.03701726e-04, 4.53487851e-04, 5.09413801e-04, 5.72236766e-04,
       6.42807312e-04, 7.22080...
       6.89261210e-01, 7.74263683e-01, 8.69749003e-01, 9.77009957e-01,
       1.09749877e+00, 1.23284674e+00, 1.38488637e+00, 1.55567614e+00,
       1.74752840e+00, 1.96304065e+00, 2.20513074e+00, 2.47707636e+00,
       2.78255940e+00, 3.12571585e+00, 3.51119173e+00, 3.94420606e+00,
       4.43062146e+00, 4.97702356e+00, 5.59081018e+00, 6.28029144e+00,
       7.05480231e+00, 7.92482898e+00, 8.90215085e+00, 1.00000000e+01])},
                   random_state=42, scoring='r2')

In [49]:
print("Best alpha:", random_search.best_params_['alpha'])
print("Best R^2:", random_search.best_score_)

Best alpha: 0.0001
Best R^2: 0.29003416755492584


Фреймворк Optuna

In [50]:
def objective(trial):
    alpha = trial.suggest_loguniform('alpha', 1e-4, 10.0)
    model = Lasso(alpha=alpha)
    score = cross_val_score(model, X_train, y_train, scoring='r2', cv=5).mean()
    return score

In [51]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)

[I 2025-04-24 08:53:53,045] A new study created in memory with name: no-name-be43ff74-30b9-432e-a748-f7460a589de9
C:\Users\andre\AppData\Local\Temp\ipykernel_7732\1532896469.py:2: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  alpha = trial.suggest_loguniform('alpha', 1e-4, 10.0)
[I 2025-04-24 08:53:53,062] Trial 0 finished with value: 0.23317968502276254 and parameters: {'alpha': 0.03279596917618485}. Best is trial 0 with value: 0.23317968502276254.
C:\Users\andre\AppData\Local\Temp\ipykernel_7732\1532896469.py:2: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  alpha = trial.suggest_loguniform('alpha', 1e-4, 10.0)
[I 2025-04-24 08:53:53,077] Trial 1 finished with value: 0

In [52]:
print("Best alpha:", study.best_params['alpha'])
print("Best R^2:", study.best_value)

Best alpha: 0.00010011172867186169
Best R^2: 0.2900341170802035


## Линейная регрессия с L2-регуляризацией

In [53]:
lasso = Lasso(alpha=0.5).fit(X_train, y_train)
y_pred = lasso.predict(X_test)
print(f'MAE: {mean_absolute_error(y_test, y_pred)}')
print(f'MSE: {mean_squared_error(y_test, y_pred)}')
print(f'RMSE: {sqrt(mean_squared_error(y_test, y_pred))}')
print(f'MAPE: {sqrt(mean_absolute_percentage_error(y_test, y_pred))}')
print(f'R^2: {lasso.score(X_test, y_test)}')
lasso.coef_

MAE: 0.6827676588294879
MSE: 0.7474827906496314
RMSE: 0.8645708708079584
MAPE: 0.3509441940851519
R^2: 0.006959429888808155


array([-0.        , -0.        ,  0.        , -0.        , -0.        ,
        0.00573309, -0.00169882, -0.        ,  0.        ,  0.        ,
        0.        ])

### Подбор гиперпараметров

In [54]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, cross_val_score
import numpy as np
import optuna

GridSearchCV

In [55]:
param_grid = {'alpha': [0.001, 0.01, 0.1, 0.5, 1, 10, 100]}

In [56]:
grid = GridSearchCV(Ridge(), param_grid, cv=5, scoring='r2')
grid.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=Ridge(),
             param_grid={'alpha': [0.001, 0.01, 0.1, 0.5, 1, 10, 100]},
             scoring='r2')

In [57]:
print("Best alpha:", grid.best_params_['alpha'])
print("Best R^2:", grid.best_score_)

Best alpha: 0.001
Best R^2: 0.2920287284310427


RandomizedSearchCV

In [58]:
param_dist = {'alpha': np.logspace(-4, 2, 100)}

In [59]:
random_search = RandomizedSearchCV(Ridge(), param_distributions=param_dist, n_iter=20, cv=5,
                                   scoring='r2', random_state=42)
random_search.fit(X_train, y_train)

RandomizedSearchCV(cv=5, estimator=Ridge(), n_iter=20,
                   param_distributions={'alpha': array([1.00000000e-04, 1.14975700e-04, 1.32194115e-04, 1.51991108e-04,
       1.74752840e-04, 2.00923300e-04, 2.31012970e-04, 2.65608778e-04,
       3.05385551e-04, 3.51119173e-04, 4.03701726e-04, 4.64158883e-04,
       5.33669923e-04, 6.13590727e-04, 7.05480231e-04, 8.11130831e-04,
       9.32603347e-04, 1.07226...
       4.03701726e+00, 4.64158883e+00, 5.33669923e+00, 6.13590727e+00,
       7.05480231e+00, 8.11130831e+00, 9.32603347e+00, 1.07226722e+01,
       1.23284674e+01, 1.41747416e+01, 1.62975083e+01, 1.87381742e+01,
       2.15443469e+01, 2.47707636e+01, 2.84803587e+01, 3.27454916e+01,
       3.76493581e+01, 4.32876128e+01, 4.97702356e+01, 5.72236766e+01,
       6.57933225e+01, 7.56463328e+01, 8.69749003e+01, 1.00000000e+02])},
                   random_state=42, scoring='r2')

In [60]:
print("Best alpha:", random_search.best_params_['alpha'])
print("Best R^2:", random_search.best_score_)

Best alpha: 0.0001
Best R^2: 0.2922107210140453


Фреймворк Optuna

In [61]:
def objective(trial):
    alpha = trial.suggest_loguniform('alpha', 1e-4, 100.0)
    model = Ridge(alpha=alpha)
    score = cross_val_score(model, X_train, y_train, scoring='r2', cv=5).mean()
    return score

In [62]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)

[I 2025-04-24 08:53:53,973] A new study created in memory with name: no-name-f8c4682f-0724-4cb3-9b84-ff003e9849c2
C:\Users\andre\AppData\Local\Temp\ipykernel_7732\1837225706.py:2: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  alpha = trial.suggest_loguniform('alpha', 1e-4, 100.0)
[I 2025-04-24 08:53:53,989] Trial 0 finished with value: 0.2921199151722071 and parameters: {'alpha': 0.0006380390610412122}. Best is trial 0 with value: 0.2921199151722071.
C:\Users\andre\AppData\Local\Temp\ipykernel_7732\1837225706.py:2: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  alpha = trial.suggest_loguniform('alpha', 1e-4, 100.0)
[I 2025-04-24 08:53:54,003] Trial 1 finished with value:

In [63]:
print("Best alpha:", study.best_params['alpha'])
print("Best R^2:", study.best_value)

Best alpha: 0.0001009266171156292
Best R^2: 0.29221066252366007


## Линейная регрессия с двумя регуляризаторами

In [64]:
from sklearn.linear_model import ElasticNet

In [65]:
base_elastic_model = ElasticNet(max_iter=100000, alpha=1.0, l1_ratio=0.5)
base_elastic_model.fit(X_train, y_train)
y_pred = base_elastic_model.predict(X_test)
print(f'MAE: {mean_absolute_error(y_test, y_pred)}')
print(f'MSE: {mean_squared_error(y_test, y_pred)}')
print(f'RMSE: {sqrt(mean_squared_error(y_test, y_pred))}')
print(f'MAPE: {sqrt(mean_absolute_percentage_error(y_test, y_pred))}')
print(f'R^2: {base_elastic_model.score(X_test, y_test)}')
base_elastic_model.coef_

MAE: 0.6827856535151661
MSE: 0.7474761758992707
RMSE: 0.8645670453465543
MAPE: 0.3509487088793306
R^2: 0.006968217670348387


array([-0.        , -0.        ,  0.        , -0.        , -0.        ,
        0.00571223, -0.00169389, -0.        ,  0.        ,  0.        ,
        0.        ])

### Подбор гиперпараметров

In [66]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, cross_val_score
import numpy as np
import optuna

GridSearchCV

In [67]:
param_grid = {
    'alpha': [0.001, 0.01, 0.1, 1, 10],
    'l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9]
}

In [68]:
grid = GridSearchCV(
    ElasticNet(max_iter=100000),
    param_grid,
    scoring='r2',
    cv=5
)
grid.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=ElasticNet(max_iter=100000),
             param_grid={'alpha': [0.001, 0.01, 0.1, 1, 10],
                         'l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9]},
             scoring='r2')

In [69]:
print("Best params:", grid.best_params_)
print("Best R^2:", grid.best_score_)

Best params: {'alpha': 0.001, 'l1_ratio': 0.1}
Best R^2: 0.2900295669129872


RandomizedSearchCV

In [70]:
param_dist = {
    'alpha': np.logspace(-4, 1, 100),
    'l1_ratio': np.linspace(0.1, 0.9, 9)
}

In [71]:
random_search = RandomizedSearchCV(
    ElasticNet(max_iter=100000),
    param_distributions=param_dist,
    n_iter=30,
    scoring='r2',
    cv=5,
    random_state=42
)
random_search.fit(X_train, y_train)

RandomizedSearchCV(cv=5, estimator=ElasticNet(max_iter=100000), n_iter=30,
                   param_distributions={'alpha': array([1.00000000e-04, 1.12332403e-04, 1.26185688e-04, 1.41747416e-04,
       1.59228279e-04, 1.78864953e-04, 2.00923300e-04, 2.25701972e-04,
       2.53536449e-04, 2.84803587e-04, 3.19926714e-04, 3.59381366e-04,
       4.03701726e-04, 4.53487851e-04, 5.09413801e-04, 5.72236766e-04,
       6....
       1.09749877e+00, 1.23284674e+00, 1.38488637e+00, 1.55567614e+00,
       1.74752840e+00, 1.96304065e+00, 2.20513074e+00, 2.47707636e+00,
       2.78255940e+00, 3.12571585e+00, 3.51119173e+00, 3.94420606e+00,
       4.43062146e+00, 4.97702356e+00, 5.59081018e+00, 6.28029144e+00,
       7.05480231e+00, 7.92482898e+00, 8.90215085e+00, 1.00000000e+01]),
                                        'l1_ratio': array([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9])},
                   random_state=42, scoring='r2')

In [72]:
print("Best params:", random_search.best_params_)
print("Best R^2:", random_search.best_score_)

Best params: {'l1_ratio': np.float64(0.1), 'alpha': np.float64(0.00025353644939701115)}
Best R^2: 0.29017210430132534


Фреймворк Optuna

In [73]:
def objective(trial):
    alpha = trial.suggest_loguniform('alpha', 1e-4, 10.0)
    l1_ratio = trial.suggest_uniform('l1_ratio', 0.1, 0.9)
    model = ElasticNet(alpha=alpha, l1_ratio=l1_ratio, max_iter=100000)
    score = cross_val_score(model, X_train, y_train, cv=5, scoring='r2').mean()
    return score

In [74]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=40)

[I 2025-04-24 08:53:55,285] A new study created in memory with name: no-name-ff5bf48a-0bbd-423e-8fa6-c94bcc8a9d1e
C:\Users\andre\AppData\Local\Temp\ipykernel_7732\4049124632.py:2: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  alpha = trial.suggest_loguniform('alpha', 1e-4, 10.0)
C:\Users\andre\AppData\Local\Temp\ipykernel_7732\4049124632.py:3: FutureWarning: suggest_uniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float instead.
  l1_ratio = trial.suggest_uniform('l1_ratio', 0.1, 0.9)
[I 2025-04-24 08:53:55,302] Trial 0 finished with value: 0.012492298327272566 and parameters: {'alpha': 1.2677254174831054, 'l1_ratio': 0.4045462724326605}. Best is trial 0 with value: 0.012492298327272566.
C:\Users\andre\AppData\Local\Temp\ipykerne

In [75]:
print("Best params:", study.best_params)
print("Best R^2:", study.best_value)

Best params: {'alpha': 0.00017457972278261434, 'l1_ratio': 0.11615835791127176}
Best R^2: 0.29015560574725197


## Полиномиальная регрессия

In [76]:
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline

In [77]:
polynomial_model = Pipeline([
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('scaler', StandardScaler()),
    ('reg', LinearRegression())
])

In [78]:
polynomial_model.fit(X_train, y_train)

Pipeline(steps=[('poly', PolynomialFeatures(include_bias=False)),
                ('scaler', StandardScaler()), ('reg', LinearRegression())])

In [79]:
y_pred = polynomial_model.predict(X_test)
print("MAE:", mean_absolute_error(y_test, y_pred))
print("R^2:", r2_score(y_test, y_pred))


MAE: 0.5527904405911641
R^2: 0.2609930674438008


### Подбор гиперпараметров

In [80]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, cross_val_score
import numpy as np
import optuna

GridSearchCV

In [2]:
pipeline = Pipeline([
    ('poly', PolynomialFeatures(include_bias=False)),
    ('scaler', StandardScaler()),
    ('reg', ElasticNet(max_iter=100000))
])

NameError: name 'Pipeline' is not defined

In [82]:
param_grid = {
    'poly__degree': [2, 3],
    'reg__alpha': [0.001, 0.01, 0.1, 1],
    'reg__l1_ratio': [0.2, 0.5, 0.8]
}

In [83]:
grid = GridSearchCV(pipeline, param_grid, cv=5, scoring='r2')
grid.fit(X_train, y_train)

p:\Python\ML1\venv\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.016e-01, tolerance: 3.113e-01
  model = cd_fast.enet_coordinate_descent(
p:\Python\ML1\venv\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 5.512e-01, tolerance: 3.177e-01
  model = cd_fast.enet_coordinate_descent(


GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('poly',
                                        PolynomialFeatures(include_bias=False)),
                                       ('scaler', StandardScaler()),
                                       ('reg', ElasticNet(max_iter=100000))]),
             param_grid={'poly__degree': [2, 3],
                         'reg__alpha': [0.001, 0.01, 0.1, 1],
                         'reg__l1_ratio': [0.2, 0.5, 0.8]},
             scoring='r2')

In [84]:
print("Best params:", grid.best_params_)
print("Best R^2:", grid.best_score_)

Best params: {'poly__degree': 2, 'reg__alpha': 0.01, 'reg__l1_ratio': 0.2}
Best R^2: 0.31657014289806007


RandomizedSearchCV

In [85]:
param_dist = {
    'poly__degree': [2, 3],
    'reg__alpha': np.logspace(-4, 1, 100),
    'reg__l1_ratio': np.linspace(0.1, 0.9, 9)
}

In [86]:
random_search = RandomizedSearchCV(
    pipeline, param_distributions=param_dist,
    n_iter=30, cv=5, scoring='r2', random_state=42
)
random_search.fit(X_train, y_train)

p:\Python\ML1\venv\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.021e+00, tolerance: 3.173e-01
  model = cd_fast.enet_coordinate_descent(
p:\Python\ML1\venv\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.719e+00, tolerance: 3.210e-01
  model = cd_fast.enet_coordinate_descent(
p:\Python\ML1\venv\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.778e+00, tolerance: 3.113e-01
  mod

RandomizedSearchCV(cv=5,
                   estimator=Pipeline(steps=[('poly',
                                              PolynomialFeatures(include_bias=False)),
                                             ('scaler', StandardScaler()),
                                             ('reg',
                                              ElasticNet(max_iter=100000))]),
                   n_iter=30,
                   param_distributions={'poly__degree': [2, 3],
                                        'reg__alpha': array([1.00000000e-04, 1.12332403e-04, 1.26185688e-04, 1.41747416e-04,
       1.59228279e-04, 1.78864953e-04, 2.00923300e-04, 2.2570197...
       1.09749877e+00, 1.23284674e+00, 1.38488637e+00, 1.55567614e+00,
       1.74752840e+00, 1.96304065e+00, 2.20513074e+00, 2.47707636e+00,
       2.78255940e+00, 3.12571585e+00, 3.51119173e+00, 3.94420606e+00,
       4.43062146e+00, 4.97702356e+00, 5.59081018e+00, 6.28029144e+00,
       7.05480231e+00, 7.92482898e+00, 8.90215085e+00, 1.00000000e+01]),
                                        'reg__l1_ratio': array([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9])},
                   random_state=42, scoring='r2')

In [87]:
print("Best params:", random_search.best_params_)
print("Best R^2:", random_search.best_score_)

Best params: {'reg__l1_ratio': np.float64(0.2), 'reg__alpha': np.float64(0.0001), 'poly__degree': 3}
Best R^2: 0.3216131021517742


Фреймворк Optuna

In [88]:
def objective(trial):
    degree = trial.suggest_int('degree', 2, 3)
    alpha = trial.suggest_loguniform('alpha', 1e-4, 10.0)
    l1_ratio = trial.suggest_uniform('l1_ratio', 0.1, 0.9)

    pipeline = Pipeline([
        ('poly', PolynomialFeatures(degree=degree, include_bias=False)),
        ('scaler', StandardScaler()),
        ('reg', ElasticNet(alpha=alpha, l1_ratio=l1_ratio, max_iter=100000))
    ])

    score = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='r2').mean()
    return score

In [ ]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=40)

[I 2025-04-24 09:25:21,463] A new study created in memory with name: no-name-b1cae3a6-76b3-4b22-83f4-ec2f81d42c63
C:\Users\andre\AppData\Local\Temp\ipykernel_7732\2030623893.py:3: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  alpha = trial.suggest_loguniform('alpha', 1e-4, 10.0)
C:\Users\andre\AppData\Local\Temp\ipykernel_7732\2030623893.py:4: FutureWarning: suggest_uniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float instead.
  l1_ratio = trial.suggest_uniform('l1_ratio', 0.1, 0.9)
[I 2025-04-24 09:25:21,657] Trial 0 finished with value: 0.04212434029954393 and parameters: {'degree': 3, 'alpha': 1.8629571271972767, 'l1_ratio': 0.1826122062032785}. Best is trial 0 with value: 0.04212434029954393.
C:\Users\andre\AppData\Local\Te

In [1]:
print("Best params:", study.best_params)
print("Best R^2:", study.best_value)

NameError: name 'study' is not defined